In [1]:
import cell2location
from cell2location.models import RegressionModel
import matplotlib.pyplot as plt
import random
import scanpy as sc
import numpy as np
import pandas as pd
import math
from anndata import AnnData
from typing import Optional

/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
def run_cell2location(
        adata_sc: AnnData,
        adata_st: AnnData,
        cell_num_per_spot: int = 5,
        celltype_key: Optional[str] = 'Cell_type',
        sc_epochs: Optional[int] = 1000,
        st_epochs: Optional[int] = 30000):

    # sc
    # prepare anndata for the regression model
    cell2location.models.RegressionModel.setup_anndata(adata=adata_sc, labels_key=celltype_key)
    # create the regression model
    mod = RegressionModel(adata_sc)
    mod.train(max_epochs=sc_epochs)

    adata_sc = mod.export_posterior(
        adata_sc, sample_kwargs={'num_samples': 1000, 'batch_size': 2500}
    )
    # export estimated expression in each cluster
    if 'means_per_cluster_mu_fg' in adata_sc.varm.keys():
        inf_aver = adata_sc.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                                             for i in adata_sc.uns['mod']['factor_names']]].copy()
    else:
        inf_aver = adata_sc.var[[f'means_per_cluster_mu_fg_{i}'
                                 for i in adata_sc.uns['mod']['factor_names']]].copy()
    inf_aver.columns = adata_sc.uns['mod']['factor_names']

    # st
    # prepare anndata for cell2location model
    cell2location.models.Cell2location.setup_anndata(adata=adata_st)
    # create and train the model
    # cell_num_per_spot = np.round(np.mean(adata_st.obs['Estimate_cell_num'])).astype(int)
    mod = cell2location.models.Cell2location(
        adata_st,
        cell_state_df=inf_aver,
        N_cells_per_location=cell_num_per_spot,
        detection_alpha=20
    )
    mod.train(max_epochs=st_epochs, batch_size=None, train_size=1)

    adata_st = mod.export_posterior(
        adata_st, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
    )

    res = adata_st.obsm['q05_cell_abundance_w_sf']
    # normalize
    res = res.div(res.sum(axis=1), axis='rows')
    column_name = res.columns.tolist()
    column_name = [column_name[i].replace('q05cell_abundance_w_sf_', '') for i in range(len(column_name))]
    res.columns = column_name

    return res

In [3]:
# demo data can be downloaded via https://drive.google.com/drive/folders/1UOBifg53QwayzZwXbGYCJSvmDc58RofD?usp=sharing
sc_adata = sc.read('sc_kidney.h5ad')
st_adata = sc.read('st_kidney.h5ad')

sc_adata.X = np.round(sc_adata.X).astype(int)
st_adata.X = np.round(st_adata.X).astype(int)

In [4]:
# here we provide a simple demo to run cell2location. To see more details about cell2location via https://github.com/BayraktarLab/cell2location
res = run_cell2location(
    adata_sc=sc_adata,
    adata_st=st_adata,
    cell_num_per_spot=5,
    celltype_key='Cell_type',
    sc_epochs=1000,
    st_epochs=30000
)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2l ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention 

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2l ...


Sampling local variables, batch:   0%|          | 0/3 [00:00<?, ?it/s]

Sampling global variables, sample:   0%|          | 0/999 [00:00<?, ?it/s]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2l ...
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, usi

Training:   0%|          | 0/30000 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30000` reached.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
/slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2loc_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /slurm/home/yrd/liaolab/baohudong/.conda/envs/cell2l ...


Sampling local variables, batch:   0%|          | 0/1 [00:00<?, ?it/s]

Sampling global variables, sample:   0%|          | 0/999 [00:00<?, ?it/s]

In [5]:
res

,Distal tubule,Intercalated cells,Principal cells,Proximal tubule,Renal corpuscle,Thin limb of Loop of Henle
AAACAAGTATCTCCCA-1,0.619070,0.004809,0.002868,0.297330,0.038485,0.037438
AAACAATCTACTAGCA-1,0.266194,0.361110,0.045568,0.289949,0.036891,0.000288
AAACAGAGCGACTCCT-1,0.098578,0.018226,0.000108,0.807581,0.075485,0.000021
AAACATTTCCCGGATT-1,0.824754,0.003418,0.016634,0.018819,0.064683,0.071693
AAACCCGAACGAAATC-1,0.237044,0.012815,0.001727,0.712091,0.028243,0.008079
...,...,...,...,...,...,...
TTGTTCAGTGTGCTAC-1,0.484029,0.015079,0.028640,0.337438,0.059242,0.075571
TTGTTGTGTGTCAAGA-1,0.599618,0.013242,0.026074,0.248303,0.049958,0.062805
TTGTTTCACATCCAGG-1,0.005919,0.000134,0.380901,0.024978,0.000365,0.587703
TTGTTTCCATACAACT-1,0.108852,0.009043,0.002109,0.872054,0.006820,0.001121


In [6]:
#res.to_csv('C2L_res_kidney.csv')